#### ***Installing libraries***

In [56]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain_core.chatopenai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch, RunnablePassthrough
import os
from dotenv import load_dotenv

load_dotenv()

llm_openai= init_chat_model(model=os.environ.get("LLM_MODEL"), temperature = 0.7)
# generation = input("Enter your generation (e.g., GenZ, Millennial, GenX): ")

In [57]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "neutral", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

In [58]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([("user", "Please categorize the movie review as positive or negative: {input}"),
("system", "You are a movie review evaluator.")])

In [59]:
# TASK - 2 [LLM]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

In [60]:
# TASK - 3 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def pydantic_json(input:llm_schema)-> str:

    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

##### ***Conditional Chain 1***

In [61]:
def defend_rev(review: str):

    prompt_def = ChatPromptTemplate.from_messages([("user", "Defend the movie against this review: {review} as to why the movie is actually good"),
    ("system", "You are someone associated with the movie, the director. You have to explain why it is a good movie. Reply within 280 characters.")])

    llm_openai= init_chat_model(model=os.environ.get("LLM_MODEL"), temperature = 0.1)

    str_parse = StrOutputParser()

    chain_def = prompt_def | llm_openai | str_parse

    result = chain_def.invoke({"review": review})

    return result

##### ***Conditional Chain 2***

In [62]:
def thank_rev(review):

    prompt_thx = ChatPromptTemplate.from_messages([("user", "Thanks the audience for the review: {review}"),
    ("system", "You are someone associated with the movie, the director. You have to thank the audience for their positive feedback. Reply within 280 characters.")])

    llm_openai= init_chat_model(model=os.environ.get("LLM_MODEL"), temperature = 0.1)

    str_parse = StrOutputParser()

    chain_thx = prompt_thx | llm_openai | str_parse

    result = chain_thx.invoke({"review": review})

    return result

##### ***Conditional Chain 3***

In [ ]:
def neutral_rev(review: str):

    prompt_def = ChatPromptTemplate.from_messages([("user", "Ask the audience to actually give a review rather than give a neutral review like this review: {review} as to why audience should actually give an opinion."),
    ("system", "You are someone associated with the movie, the director. You have to explain why it is a important to have a critical review for a good movie like this. Reply within 280 characters.")])

    llm_openai= init_chat_model(model=os.environ.get("LLM_MODEL"), temperature = 0.1)

    str_parse = StrOutputParser()

    chain_def = prompt_def | llm_openai | str_parse

    result = chain_def.invoke({"review": review})

    return result

##### ***Orchestrator***

In [64]:
thank_chain_runnable = RunnableLambda(thank_rev)
defend_chain_runnable = RunnableLambda(defend_rev)
neutral_chain_runnable = RunnableLambda(neutral_rev)

classify = RunnableParallel({
    "flag" : prompt_template | llm_structured_output | pydantic_json_lambda,
    "review": RunnableLambda(lambda x: x["input"])
})

conditional_chain = RunnableBranch(
    (lambda x: x["flag"] == "negative", RunnableLambda(lambda x: x["review"]) | defend_chain_runnable),
    (lambda x: x["flag"] == "neutral", RunnableLambda(lambda x: x["review"]) | defend_chain_runnable),
     RunnableLambda(lambda x: x["review"]) | thank_chain_runnable,
)

final_orch = classify | conditional_chain

In [67]:
final_orch.invoke({"input": "I am neutral with this KGF movie"})

"As KGF's director I crafted a raw, lived-in world of ambition, grit and sacrifice. Bold production design, striking cinematography, intense action choreography, a haunting score and a visceral lead performance give it emotional depth beyond spectacle."